# 13장. NumPy로 TF-IDF 구현하기

| 항목 | 내용 |
|---|---|
| 예상 시간 | 4시간 |
| 선수 장 | 4장, 11장, 12장 |
| 이 장의 결과물 | `NumpyTfidf` 클래스 (`src/tfidf.py`), 손계산 기반 테스트 |
| 과제 연결 | "NumPy를 사용하여 TF-IDF 행렬을 계산하는 함수를 작성한다" |

**학습 목표**
- `fit`, `count_matrix`, `transform`으로 나눈 TF-IDF 클래스를 설계할 수 있다.
- 문서 빈도, IDF, TF, 정규화를 루프 없는 NumPy 연산으로 계산할 수 있다.
- 빈 문서, OOV, 어휘가 비는 경우를 처리할 수 있다.
- 12장 손계산 값으로 구현을 테스트할 수 있다.
- 실제 데이터에서 문서별 상위 가중치 단어를 뽑아 해석할 수 있다.

> 이 장의 첫 셀에는 데이터 로딩과 전처리만 있고 TF-IDF 클래스는 없습니다. 클래스는 이 장에서 직접 만듭니다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

# ── 8~10장에서 만든 전처리 파이프라인 (src/preprocess.py로 옮길 내용) ──
from konlpy.tag import Okt
from nltk.corpus import stopwords

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}
EN_STOP = frozenset(stopwords.words("english"))
DOMAIN_STOPWORDS_EN = frozenset({
    "would", "could", "might", "may", "also", "like", "get", "got", "go", "going", "know", "think",
    "see", "say", "make", "want", "look", "one", "well", "much", "many", "even", "way", "something",
    "anyone", "since", "still", "really", "right", "another", "cant",
})
STOPWORDS_KO = frozenset({
    "하다", "되다", "있다", "없다", "이다", "않다",
    "등", "것", "수", "중", "및", "때", "후", "내", "듯", "제", "개", "의", "로", "종합",
})
_okt = Okt()


def clean_en(text: str) -> str:
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


class EnglishPreprocessor:
    def __init__(self, use_domain_stopwords: bool = True, min_len: int = 2, max_len: int = 20):
        self.stopwords = EN_STOP | (DOMAIN_STOPWORDS_EN if use_domain_stopwords else frozenset())
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for tok in clean_en(text).split():
            if tok in self.stopwords:
                continue
            tok = tok.removesuffix("'s").replace("'", "")
            if self.min_len <= len(tok) <= self.max_len and tok not in self.stopwords:
                tokens.append(tok)
        return tokens


class KoreanPreprocessor:
    def __init__(self, tokenizer: str = "okt", use_stopwords: bool = True):
        if tokenizer not in ("okt", "whitespace"):
            raise ValueError(f"tokenizer는 'okt' 또는 'whitespace': {tokenizer}")
        self.tokenizer = tokenizer
        self.stopwords = STOPWORDS_KO if use_stopwords else frozenset()

    def tokenize(self, text: str) -> list[str]:
        if self.tokenizer == "whitespace":
            return clean_ko(text).lower().split()
        return [w.lower() for w, p in _okt.pos(clean_ko(text), norm=True, stem=True)
                if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]

    def __call__(self, text: str) -> list[str]:
        return [t for t in self.tokenize(text) if t not in self.stopwords and not t.isdigit()]

# ── 로드 + 전처리를 한 번에 (src/corpus.py로 옮길 내용) ──
MIN_TOKENS = {"en": 5, "ko": 2}   # 전처리 후 토큰이 이보다 적은 문서는 뺀다 (짧은 답글, 주제어 없는 제목)
MIN_DF = {"en": 3, "ko": 2}       # 어휘에 넣을 최소 문서 빈도 (11장)


def load_corpus(lang: str, tokenizer: str = "okt") -> pd.DataFrame:
    """열: text, label, tokens. lang은 "en" 또는 "ko"."""
    df = load_english() if lang == "en" else load_korean()
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor(tokenizer)
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df

# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

import numpy as np
import pandas as pd

STOP = {"the", "a", "into", "our", "in", "new"}
FILTERED = [[t for t in doc.lower().strip(".!").split() if t not in STOP] for doc in MINI_DOCS]
np.set_printoptions(precision=4, suppress=True, linewidth=120)

import time

## 13.1 설계

2장의 `fit`/`transform` 패턴을 따릅니다. sklearn의 `TfidfVectorizer`와 같은 이름과 의미를 쓰면 14장 비교가 쉬워집니다.

| 구분 | 이름 | 뜻 |
|---|---|---|
| 설정값 (`__init__`) | `tf` | `"raw"`(횟수) 또는 `"log"`(1 + ln 횟수, sklearn `sublinear_tf=True`) |
| | `smooth_idf` | True면 $\ln\frac{1+n}{1+\mathrm{df}} + 1$, False면 $\ln\frac{n}{\mathrm{df}} + 1$ |
| | `norm` | `"l2"` 또는 `None` |
| | `min_df`, `max_df` | 어휘에 넣을 문서 빈도 범위 (11장). 정수는 개수, 실수는 비율 |
| 학습 결과 (`fit` 후) | `vocabulary_` | 단어 → 열 번호 (정렬된 순서) |
| | `terms_` | 열 번호 → 단어 배열 |
| | `n_docs_` | 학습 문서 수 $n$ |
| | `df_`, `idf_` | 단어별 문서 빈도와 IDF (길이 V 배열) |
| 메서드 | `fit(docs)` | 어휘·DF·IDF 학습, `self` 반환 |
| | `count_matrix(docs)` | (n, V) 횟수 행렬 |
| | `transform(docs)` | (n, V) TF-IDF 행렬 |
| | `fit_transform(docs)` | 둘을 한 번에 |

입력 `docs`는 **이미 전처리된 토큰 리스트의 리스트**입니다. 전처리(10장)와 벡터화를 분리해 두면, 전처리를 바꿔도 TF-IDF 코드는 그대로 쓸 수 있습니다.

## 13.2 fit: 어휘, DF, IDF

먼저 함수 하나로 써 보고, 나중에 클래스로 옮깁니다.

In [2]:
def fit_vocab_idf(docs, min_df=1, max_df=1.0, smooth_idf=True):
    n = len(docs)
    df = {}
    for doc in docs:
        for t in set(doc):                              # 문서마다 한 번만 센다 → 문서 빈도
            df[t] = df.get(t, 0) + 1
    max_count = max_df * n if isinstance(max_df, float) else max_df   # 실수는 비율, 정수는 개수
    terms = sorted(t for t, c in df.items() if min_df <= c <= max_count)
    vocabulary = {t: i for i, t in enumerate(terms)}
    df_arr = np.array([df[t] for t in terms], dtype=np.float64)
    if smooth_idf:
        idf = np.log((1.0 + n) / (1.0 + df_arr)) + 1.0
    else:
        idf = np.log(n / df_arr) + 1.0
    return vocabulary, df_arr, idf


vocabulary, df_arr, idf = fit_vocab_idf(FILTERED)
print("어휘:", list(vocabulary))
print("df  :", df_arr)
print("idf :", idf)

어휘: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']
df  : [1. 1. 2. 1. 2. 2. 1. 1. 1. 3. 1. 1. 1. 1. 1.]
idf : [2.0986 2.0986 1.6931 2.0986 1.6931 1.6931 2.0986 2.0986 2.0986 1.4055 2.0986 2.0986 2.0986 2.0986 2.0986]


- DF는 단어가 수만 개라도 문서를 한 번만 훑으면 됩니다. `set(doc)`로 문서 안의 중복을 없앤 뒤 셉니다.
- `1.0 + n`처럼 실수로 써서 정수 나눗셈 문제를 피합니다. `np.log`는 배열 전체에 한 번에 적용됩니다(4장).
- 12장 손계산과 같은지 확인합니다. df=1 → 2.0986, df=2 → 1.6931, df=3 → 1.4055.

## 13.3 count_matrix와 transform

`count_matrix`는 11장의 `np.add.at` 방식입니다. `transform`은 **TF → IDF 곱하기 → 정규화** 순서입니다.

In [3]:
def count_matrix(docs, vocabulary):
    rows, cols = [], []
    for i, doc in enumerate(docs):
        for t in doc:
            j = vocabulary.get(t)
            if j is not None:                            # OOV는 버린다
                rows.append(i)
                cols.append(j)
    counts = np.zeros((len(docs), len(vocabulary)), dtype=np.float64)
    np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
    return counts


def transform(docs, vocabulary, idf, tf="raw", norm="l2"):
    x = count_matrix(docs, vocabulary)
    if tf == "log":
        mask = x > 0
        x[mask] = 1.0 + np.log(x[mask])                 # 0인 칸은 계산하지 않는다 (4장 log(0) 함정)
    x *= idf                                             # (n, V) * (V,): 열마다 IDF 곱하기
    if norm == "l2":
        norms = np.linalg.norm(x, axis=1, keepdims=True) # (n, 1)
        x /= np.where(norms > 0, norms, 1.0)             # 빈 문서는 0 벡터로 둔다
    return x


X = transform(FILTERED, vocabulary, idf)
print("shape:", X.shape)
print("doc 0:", {w: round(X[0, j], 4) for w, j in vocabulary.items() if X[0, j] > 0})

shape: (5, 15)
doc 0: {'launched': np.float64(0.5568), 'rocket': np.float64(0.6902), 'space': np.float64(0.4622)}


12장 손계산과 같습니다(`rocket` 0.6902, `launched` 0.5568, `space` 0.4622).

> **제자리 연산(`*=`, `/=`)**: `x = x * idf`는 새 배열을 하나 더 만들지만 `x *= idf`는 기존 배열을 고칩니다. 영어 데이터의 행렬은 약 500MB이므로, 복사본을 만들지 않는 것이 메모리를 크게 아낍니다.

## 13.4 클래스로 묶기

위 함수들을 2장의 규칙(설정값은 `__init__`, 학습 결과는 이름 끝에 `_`)에 맞춰 클래스로 옮깁니다. **이 셀이 `src/tfidf.py`가 되고, 15–20장이 이 클래스를 씁니다.**

In [4]:
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)


vec = NumpyTfidf()
X = vec.fit_transform(FILTERED)
print("함수 버전과 같은가:", np.allclose(X, transform(FILTERED, vocabulary, idf)))

함수 버전과 같은가: True


## 13.5 경계 상황

실제 데이터에서는 "평범하지 않은" 입력이 반드시 들어옵니다.

In [5]:
vec = NumpyTfidf().fit(FILTERED)

# ① 학습 때 없던 단어(OOV)는 버려진다
q = vec.transform([["space", "moon", "rocket"]])
print("OOV 'moon' 포함 쿼리:", {w: round(q[0, j], 4) for w, j in vec.vocabulary_.items() if q[0, j] > 0})

# ② 단어가 하나도 없는(또는 전부 OOV인) 문서는 0 벡터
z = vec.transform([[], ["moon", "mars"]])
print("빈 문서, 전부 OOV  :", z.sum(axis=1), "nan 있음?", np.isnan(z).any())

# ③ min_df가 너무 크면 어휘가 비어 버린다
empty = NumpyTfidf(min_df=10).fit(FILTERED)
print("min_df=10 어휘 수  :", len(empty.vocabulary_), "→ transform shape", empty.transform(FILTERED).shape)

OOV 'moon' 포함 쿼리: {'rocket': np.float64(0.8309), 'space': np.float64(0.5565)}
빈 문서, 전부 OOV  : [0. 0.] nan 있음? False
min_df=10 어휘 수  : 0 → transform shape (5, 0)


| 상황 | 처리 | 이유 |
|---|---|---|
| OOV 단어 | 버린다 | 학습한 어휘에 열이 없다. 검색어에 OOV만 있으면 0 벡터가 된다(16장) |
| 빈 문서 / 전부 OOV | 0 벡터 | 노름이 0이라 나누면 `nan`이 된다(4장). 0으로 나누기를 피해 0으로 둔다 |
| 어휘가 비는 경우 | 열이 0개인 행렬 | 계산은 되지만 쓸모가 없다. 실습 2에서 에러를 내도록 고쳐 본다 |

## 13.6 테스트: 12장 손계산과 비교

12장에서 손으로 계산한 값을 정답으로 테스트를 만듭니다. 이 함수들을 그대로 `tests/test_tfidf.py`로 옮기면 pytest가 실행합니다(2장).

In [6]:
def test_idf_matches_hand_calculation():
    vec = NumpyTfidf().fit(FILTERED)
    hand = {"rocket": 2.0986, "launched": 1.6931, "space": 1.4055}   # df = 1, 2, 3 (n = 5)
    for w, v in hand.items():
        assert abs(vec.idf_[vec.vocabulary_[w]] - v) < 1e-4


def test_doc0_vector_matches_hand_calculation():
    vec = NumpyTfidf()
    X = vec.fit_transform(FILTERED)
    hand = {"rocket": 0.6902, "launched": 0.5568, "space": 0.4622}
    for w, v in hand.items():
        assert abs(X[0, vec.vocabulary_[w]] - v) < 1e-4
    assert np.count_nonzero(X[0]) == 3


def test_rows_have_unit_norm():
    X = NumpyTfidf().fit_transform(FILTERED)
    assert np.allclose(np.linalg.norm(X, axis=1), 1.0)


def test_vocabulary_is_sorted():
    vec = NumpyTfidf().fit(FILTERED)
    assert list(vec.vocabulary_) == sorted(vec.vocabulary_)


def test_oov_and_empty_documents():
    vec = NumpyTfidf().fit(FILTERED)
    X = vec.transform([[], ["moon"]])
    assert X.shape == (2, len(vec.vocabulary_)) and not X.any() and not np.isnan(X).any()


def test_log_tf():
    vec = NumpyTfidf(tf="log", norm=None).fit(FILTERED)
    x = vec.transform([["space", "space"]])[0]
    j = vec.vocabulary_["space"]
    assert abs(x[j] - (1 + np.log(2)) * vec.idf_[j]) < 1e-12


def test_min_df_filters_rare_words():
    vec = NumpyTfidf(min_df=2).fit(FILTERED)
    assert set(vec.vocabulary_) == {"baseball", "game", "launched", "space"}


for name, f in list(globals().items()):
    if name.startswith("test_") and callable(f):
        try:
            f()
            print("✅", name)
        except AssertionError:
            print("❌", name)

✅ test_idf_matches_hand_calculation
✅ test_doc0_vector_matches_hand_calculation
✅ test_rows_have_unit_norm
✅ test_vocabulary_is_sorted
✅ test_oov_and_empty_documents
✅ test_log_tf
✅ test_min_df_filters_rare_words


## 13.7 실제 데이터에 적용하기

영어(min_df=3)와 한국어(min_df=2) 전체에 적용합니다. 영어 행렬은 약 500MB입니다.

In [7]:
en = load_corpus("en")
ko = load_corpus("ko")

t0 = time.perf_counter()
en_vec = NumpyTfidf(min_df=MIN_DF["en"])
X_en = en_vec.fit_transform(en["tokens"].tolist())
print(f"영어  : {X_en.shape}, {X_en.nbytes / 1024**2:.0f} MB, {time.perf_counter() - t0:.1f}초")

t0 = time.perf_counter()
ko_vec = NumpyTfidf(min_df=MIN_DF["ko"])
X_ko = ko_vec.fit_transform(ko["tokens"].tolist())
print(f"한국어: {X_ko.shape}, {X_ko.nbytes / 1024**2:.0f} MB, {time.perf_counter() - t0:.1f}초")
print("모든 행의 노름이 1 또는 0인가:", np.all(np.isclose(np.linalg.norm(X_ko, axis=1), 1) | (X_ko.sum(axis=1) == 0)))

영어  : (4646, 13387), 475 MB, 0.3초
한국어: (7000, 6332), 338 MB, 0.1초
모든 행의 노름이 1 또는 0인가: True


### IDF로 본 어휘

In [8]:
for name, vec in [("영어", en_vec), ("한국어", ko_vec)]:
    order = np.argsort(vec.idf_, kind="stable")
    print(f"{name} IDF 가장 작은 단어 (흔함):", [(t, round(v, 2)) for t, v in zip(vec.terms_[order[:8]], vec.idf_[order[:8]])])

영어 IDF 가장 작은 단어 (흔함): [('time', np.float64(2.8)), ('use', np.float64(2.97)), ('good', np.float64(2.99)), ('first', np.float64(3.01)), ('thanks', np.float64(3.06)), ('last', np.float64(3.09)), ('two', np.float64(3.11)), ('game', np.float64(3.11))]
한국어 IDF 가장 작은 단어 (흔함): [('대통령', np.float64(4.22)), ('한국', np.float64(4.39)), ('美', np.float64(4.57)), ('北', np.float64(4.64)), ('삼성', np.float64(4.68)), ('위', np.float64(4.69)), ('게시판', np.float64(4.77)), ('출시', np.float64(4.79))]


IDF가 가장 작은 단어는 불용어를 지운 뒤에도 가장 흔하게 남은 단어들입니다. IDF가 가장 큰 단어는 문서 빈도가 `min_df`와 같은 단어들이라 모두 같은 값을 가지므로 따로 보지 않습니다.

### 문서별 상위 가중치 단어
TF-IDF가 문서를 어떻게 "요약"하는지 봅니다. 카테고리마다 한 문서씩 고릅니다.

In [9]:
def top_terms(vec, X, i, k=8):
    row = X[i]
    order = np.argsort(-row, kind="stable")[:k]
    return [(vec.terms_[j], round(float(row[j]), 3)) for j in order if row[j] > 0]


for label in sorted(en["label"].unique()):
    i = int(en.index[en["label"] == label][0])
    print(f"[{label}] 토큰 {len(en.loc[i, 'tokens'])}개")
    print("   ", top_terms(en_vec, X_en, i))

[comp.graphics] 토큰 21개
    [('trillion', 0.667), ('planes', 0.362), ('graphics', 0.325), ('bits', 0.305), ('jumped', 0.218), ('apollo', 0.165), ('board', 0.151), ('colors', 0.145)]
[comp.windows.x] 토큰 73개
    [('cursor', 0.558), ('cursors', 0.351), ('defining', 0.223), ('busy', 0.209), ('complicated', 0.194), ('shown', 0.175), ('widgets', 0.173), ('expected', 0.164)]
[rec.sport.baseball] 토큰 18개
    [('lankford', 0.555), ('injury', 0.457), ('struggling', 0.323), ('slight', 0.262), ('plate', 0.239), ('cubs', 0.227), ('saw', 0.2), ('ray', 0.2)]
[rec.sport.hockey] 토큰 44개
    [('trade', 0.299), ('bruins', 0.292), ('lost', 0.261), ('middleton', 0.234), ('ratelle', 0.234), ('esposito', 0.214), ('wing', 0.184), ('brad', 0.18)]
[sci.space] 토큰 65개
    [('landings', 0.45), ('henry', 0.244), ('dc', 0.223), ('require', 0.211), ('blunt', 0.162), ('reuse', 0.162), ('descent', 0.158), ('horizontally', 0.158)]


In [10]:
for label in sorted(ko["label"].unique()):
    i = int(ko.index[ko["label"] == label][0])
    print(f"[{label}] {ko.loc[i, 'text']}")
    print("   ", top_terms(ko_vec, X_ko, i, k=6))

[IT과학] KT 데이터 공유 앱 Y데이터박스 가입자 10만 돌파
    [('데이터', 0.56), ('y', 0.382), ('박스', 0.369), ('공유', 0.334), ('가입자', 0.312), ('앱', 0.289)]
[경제] 서울신라호텔 패션브랜드 협업 디저트 매출 20%↑
    [('패션', 0.428), ('협업', 0.428), ('신라', 0.414), ('브랜드', 0.38), ('호텔', 0.369), ('매출', 0.335)]
[사회] 셀카·합성영상은 성폭력처벌법 적용 안 돼…보완책 시급
    [('셀카', 0.336), ('합성', 0.328), ('보완', 0.32), ('성폭력', 0.32), ('시급', 0.32), ('처벌', 0.314)]
[생활문화] 내주날씨 수·금요일에 중부지방 비 또는 눈
    [('중부지방', 0.559), ('내주', 0.467), ('눈', 0.42), ('날씨', 0.394), ('비', 0.372)]
[세계] 수괴 사망에도 IS 위협 여전…충성맹세에 말리 테러 배후 자처종합
    [('말리', 0.374), ('배후', 0.374), ('자처', 0.374), ('충성', 0.374), ('여전', 0.352), ('is', 0.311)]
[스포츠] 아시안게임 토너먼트 앞둔 김학범호 아무리 강조해도 모...
    [('모', 0.453), ('김학범', 0.419), ('강조', 0.402), ('아시안', 0.351), ('앞두다', 0.346), ('호', 0.337)]
[정치] 朴대통령 지방·중앙이 힘모아 위기를 기회로
    [('모으다', 0.422), ('지방', 0.393), ('기회', 0.385), ('힘', 0.378), ('중앙', 0.366), ('위기', 0.333)]


상위 단어만 봐도 대부분 어느 카테고리인지 짐작할 수 있습니다. **TF-IDF 벡터가 문서의 주제 정보를 담고 있다**는 뜻이고, 이것이 검색(15–16장)과 분류(17–20장)가 동작하는 근거입니다. 반대로 상위 단어가 주제와 상관없어 보이는 문서는 나중에 오분류될 가능성이 큽니다(20장).

In [11]:
del X_en   # 큰 행렬은 다 쓰면 지워서 메모리를 돌려준다

## ✍️ 실습 1: 단어 하나의 가중치 추적

한국어 데이터에서 단어 `"北"`이 나온 문서들을 찾아, 그 문서들에서 `"北"`의 TF-IDF 값의 평균과 최댓값을 구하세요. 그리고 값이 가장 큰 문서의 제목과 토큰 수를 출력하고, 왜 그 문서에서 값이 컸는지 설명해 보세요.

In [12]:
# 여기에 작성하세요

**정답**

In [13]:
j = ko_vec.vocabulary_["北"]
col = X_ko[:, j]
has = np.nonzero(col)[0]
print(f"'北' 문서 수 {len(has)}, IDF {ko_vec.idf_[j]:.3f}, 평균 {col[has].mean():.3f}, 최대 {col.max():.3f}")
best = int(col.argmax())
print("최대 문서:", ko.loc[best, "text"], "| 토큰:", ko.loc[best, "tokens"])

'北' 문서 수 183, IDF 4.639, 평균 0.244, 최대 0.506
최대 문서: 유엔 北제재 더민주 北이 자초한 것…동북아 안정되길 | 토큰: ['유엔', '北', '제재', '더', '민주', '北', '이', '자초', '동북아', '안정']


이 문서는 `北`이 두 번 나와 TF가 2입니다. 또 TF-IDF는 L2 정규화로 문서 안에서 나눠 가지는 값이라, **토큰이 적은 짧은 문서**일수록 한 단어가 차지하는 비율이 커서 값이 큽니다. 같은 단어도 문서에 따라 가중치가 다릅니다.

## ✍️ 실습 2: 어휘가 비면 에러 내기

`NumpyTfidf`를 상속한 `StrictTfidf`를 만들어, `fit` 결과 어휘가 비어 있으면 `ValueError("min_df/max_df 조건을 만족하는 단어가 없습니다.")`를 내게 하세요. 미니 코퍼스에 `min_df=10`으로 확인합니다.

In [14]:
# 여기에 작성하세요

**정답**

In [15]:
class StrictTfidf(NumpyTfidf):
    def fit(self, docs):
        super().fit(docs)
        if not self.vocabulary_:
            raise ValueError("min_df/max_df 조건을 만족하는 단어가 없습니다.")
        return self


try:
    StrictTfidf(min_df=10).fit(FILTERED)
except ValueError as e:
    print("ValueError:", e)
print("정상 설정:", len(StrictTfidf(min_df=2).fit(FILTERED).vocabulary_), "단어")

ValueError: min_df/max_df 조건을 만족하는 단어가 없습니다.
정상 설정: 4 단어


조용히 빈 행렬을 돌려주는 것보다, 잘못된 설정을 **바로 알려 주는** 편이 디버깅에 유리합니다. 프로젝트의 `src/tfidf.py`처럼 기본 클래스에 이 검사를 넣어도 됩니다.

## 🧐 퀴즈

**Q1.** `fit`과 `transform`을 나누는 이유를 이 장의 클래스로 설명해 보세요.
<details><summary>▶ 정답 보기</summary>

`fit`이 학습 문서로 어휘, DF, IDF를 계산해 저장하고, `transform`은 저장된 값으로 어떤 문서든(학습 문서, 검색어, 테스트 문서) 같은 열 구성의 벡터로 바꿉니다. 새 문서마다 다시 fit하면 어휘와 IDF가 달라져 벡터를 비교할 수 없습니다.
</details>

**Q2.** `x *= self.idf_`에서 `x`의 shape이 `(4646, 13387)`일 때 `idf_`의 shape은? 어떤 규칙으로 계산되나요?
<details><summary>▶ 정답 보기</summary>

`(13387,)`입니다. 브로드캐스팅으로 모든 행에 같은 IDF 벡터가 곱해져, 열(단어)마다 그 단어의 IDF가 곱해집니다.
</details>

**Q3.** sublinear TF를 계산할 때 `np.where(x > 0, 1 + np.log(x), 0)` 대신 마스크를 쓰는 이유는?
<details><summary>▶ 정답 보기</summary>

`np.where`는 양쪽 값을 모두 먼저 계산하므로 0인 칸에서도 `log(0)`을 계산해 경고가 납니다. 마스크로 0이 아닌 칸만 계산하면 경고가 없습니다(4장).
</details>

**Q4.** 같은 단어의 TF-IDF 값이 문서마다 다른 이유는?
<details><summary>▶ 정답 보기</summary>

IDF는 단어마다 하나지만, TF(문서 안 횟수)가 다르고, L2 정규화로 문서 안의 다른 단어들과 크기를 나눠 갖기 때문입니다. 짧은 문서에서는 한 단어의 비중이 커집니다.
</details>

## ✅ 체크포인트
- [ ] `NumpyTfidf`의 설정값, 학습 결과, 메서드를 설명할 수 있다.
- [ ] DF, IDF, TF, 정규화를 루프 없는 NumPy 코드로 쓸 수 있다.
- [ ] 빈 문서, OOV, 빈 어휘를 처리할 수 있다.
- [ ] 손계산 값으로 테스트를 작성할 수 있다.
- [ ] 문서별 상위 가중치 단어를 뽑아 해석할 수 있다.

## 🔗 과제 연결과 다음 장
- `NumpyTfidf`는 `src/tfidf.py`, 테스트는 `tests/test_tfidf.py`가 됩니다. 과제 요구사항 "NumPy를 사용한 TF-IDF 직접 구현"의 핵심입니다.
- README "TF-IDF 구현 과정"에는 13.2~13.4의 단계(어휘 정렬 → DF·IDF → `np.add.at` 카운트 → 브로드캐스팅 곱 → L2 정규화)를 씁니다.
- **14장. sklearn으로 검증하기**에서 이 구현이 `TfidfVectorizer`와 1e-6 이내로 같은지 확인합니다.